# Phase 1: Data extraction

Builds the 100 m feature stack for each pre-monsoon season (Mar–May 2023, 2024, 2025) from
Google Earth Engine and downloads it straight into `data/raw/` (no Google Drive round-trip, D-020).

| Band group | Source | Logic in `src/mirageai/gee_extract.py` |
|---|---|---|
| LST (target) + observation count | Landsat 8/9 C2 L2 `ST_B10`, `QA_PIXEL` cloud mask | `landsat_lst` |
| NDVI, NDBI, MNDWI | Sentinel-2 SR, `SCL` mask, median composite | `s2_indices` |
| 7 land-cover fractions, distance to water | ESA WorldCover v200 | `worldcover_binary`, `distance_to_water` |
| Elevation, slope | SRTM, slope computed in UTM (D-016) | `terrain` |

Every band is aggregated by area-weighted mean onto the 100 m grid by `to_grid`.

In [1]:
import pandas as pd
from mirageai import config, results as R

pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)

## Re-running the extraction (optional)

Needs an Earth Engine account (see README → Earth Engine setup). It is off by default: the
pipeline starts from the CSVs in `data/raw/`, and a re-run reproduced them exactly.

In [2]:
RUN_EXTRACTION = False  # set True to re-download data/raw from Earth Engine

if RUN_EXTRACTION:
    from mirageai import gee_extract as g
    g.init_ee()
    for year in config.YEARS:
        print(g.extract_year(year))

## Sanity checks (from `scripts/pipeline/run_phase2.py`)

Driver signs and the core-versus-ring contrast are stable across all three years: the city core
is **cooler** than the outer ring, the first sign of the inverted daytime heat island.

In [3]:
R.load_table("phase1_cross_year_checks")

,year,core_LST,ring_LST,core_minus_ring,core_NDVI,ring_NDVI,corr_NDVI,corr_NDBI,corr_frac_built,corr_frac_tree,corr_frac_grass,corr_MNDWI
0,2023,43.348,46.850,-3.501,0.347,0.258,-0.434,0.662,-0.193,-0.499,0.595,-0.305
1,2024,41.786,45.031,-3.245,0.349,0.243,-0.451,0.634,-0.187,-0.486,0.591,-0.275
2,2025,41.860,45.673,-3.813,0.348,0.248,-0.427,0.613,-0.196,-0.488,0.594,-0.271


![LST map 2024](../outputs/figures/01_lst_map_2024.png)

![NDVI map 2024](../outputs/figures/01_ndvi_map_2024.png)